# NDVI: обучение и проверка на Kaggle

1. Загрузите `ndvi_best_solution.zip` как **private Kaggle Dataset** и добавьте его к notebook. Если Kaggle распакует ZIP автоматически, это тоже поддерживается.
2. Settings → Accelerator → GPU T4 x2 или P100; Internet → On. Используется одна карта, вторая не объединяет VRAM автоматически.
3. Run All. Модель пишет прогресс каждые 30 секунд. Готовый результат: `/kaggle/working/submission.csv`.

Резервный CPU-кандидат уже обучен на новой версии test. Код сверяет SHA256 входов. Старый test не нужен.

В этом notebook запускаются только ML/DL и пакетная аналитика. Официальная метрика станет известна после загрузки CSV организатору.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, shutil, json, time

input_root = Path('/kaggle/input')
working = Path('/kaggle/working')
working.mkdir(exist_ok=True)
extracted = sorted(input_root.rglob('kaggle_run.py'))
archives = sorted(input_root.rglob('ndvi_best_solution.zip'))
if len(extracted) == 1:
    original = extracted[0].parent
    ROOT = working / 'ndvi_solution'
    if not ROOT.exists():
        shutil.copytree(original, ROOT)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            dest = (working / name).resolve()
            if not dest.is_relative_to(working.resolve()):
                raise ValueError('Некорректный путь в ZIP')
        z.extractall(working)
    ROOT = working / 'ndvi_solution'
else:
    raise RuntimeError('Добавьте ровно один dataset с ndvi_best_solution.zip или распакованным ndvi_solution.')
assert (ROOT / 'data/test_features.csv').exists()
os.chdir(ROOT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'])
# GPU-версия torch из Kaggle сохраняется; CPU wheel не устанавливаем.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-foundation.txt'])
import torch
print('Python:', sys.version.split()[0], 'torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Project:', ROOT)


## Бюджет

Задайте фактически оставшееся время. Значение ниже ограничивает **этот запуск**, время установки пакетов и загрузки данных в него не входит. Оставьте не менее 30–45 минут до дедлайна на скачивание и проверку submission.

Сначала проходит CV деревьев и сохраняется полноценный tree-кандидат. Затем TabICLv2 проверяется на тех же скрытых точках. Его вес определяется по development OOF, audit не выбирает победителя. При ошибке или timeout резервный результат остаётся доступен.


In [ ]:
BUDGET_MINUTES = 300
ROUNDS = 3
ITERATIONS = 1800
SKIP_FOUNDATION = False
RUN_DIR = 'runs/kaggle'

cmd = [sys.executable, '-u', 'kaggle_run.py', '--budget-minutes', str(BUDGET_MINUTES),
       '--rounds', str(ROUNDS), '--iterations', str(ITERATIONS), '--run-dir', RUN_DIR]
if SKIP_FOUNDATION:
    cmd.append('--skip-foundation')
# Повторный запуск с теми же параметрами продолжает сохранённые этапы.
# Для других параметров задайте новый RUN_DIR.
subprocess.run(cmd, check=True)


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, FileLink

submission_path = working / 'submission.csv'
if not submission_path.exists():
    submission_path = working / 'submission_cpu_reference.csv'
sub = pd.read_csv(submission_path)
test = pd.read_csv(ROOT / 'data/test_features.csv')
mask = test.is_synthetic_gap.astype(str).str.lower().eq('true')
expected = test.loc[mask, ['anon_polygon_id', 'date']]
assert list(sub.columns) == ['anon_polygon_id', 'date', 'primary_ndvi_pred']
assert len(sub) == len(expected) == 2323
assert not sub.duplicated(['anon_polygon_id', 'date']).any()
assert np.isfinite(sub.primary_ndvi_pred).all()
joined = sub.merge(expected, on=['anon_polygon_id', 'date'], how='outer', indicator=True, validate='one_to_one')
assert joined['_merge'].eq('both').all()
print('CSV проверен:', submission_path, 'строк:', len(sub))
selection = working / 'selected_submission.json'
if selection.exists():
    print(selection.read_text())
report = ROOT / RUN_DIR / 'reports/metrics.json'
if report.exists():
    m = json.loads(report.read_text())
    print('Development:', m.get('development_ensemble'))
    print('Audit:', m.get('audit_ensemble'))
    print('Weights:', m.get('weights'))
display(sub.head())
display(FileLink(str(submission_path)))


In [ ]:
# Архив результатов без пересоздаваемого cache и четырёх лишних копий CV-весов TabICL.
result_zip = working / 'ndvi_kaggle_results.zip'
run_path = ROOT / RUN_DIR
with zipfile.ZipFile(result_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as z:
    for f in run_path.rglob('*'):
        if not f.is_file():
            continue
        rel = f.relative_to(run_path)
        if 'cache' in rel.parts:
            continue
        if any(part.startswith('tabicl_fold_') for part in rel.parts) and f.name == 'fitted.pkl':
            continue
        z.write(f, 'run/' + str(rel))
    z.write(submission_path, 'submission.csv')
    if selection.exists():
        z.write(selection, 'selected_submission.json')
        selected_dir = Path(json.loads(selection.read_text())['selected_model_directory'])
        for sf in selected_dir.rglob('*'):
            if sf.is_file():
                z.write(sf, 'selected_model/' + str(sf.relative_to(selected_dir)))
print(result_zip, 'MB:', round(result_zip.stat().st_size / 1e6, 1))
display(FileLink(str(result_zip)))


После запуска сохраните `submission.csv` и `ndvi_kaggle_results.zip`. Для разбора результата нужны `metrics.json`, `oof_predictions.csv`, `experiments.csv` и логи. Не сообщайте RMSE audit как официальный балл test.

Если предобученные веса не скачиваются: проверьте Internet в settings. Можно отдельно добавить публичный checkpoint `jingang/TabICL/tabicl-regressor-v2-20260212.ckpt` как Kaggle Dataset и передать `--tabicl-model-path` при ручном запуске `ndvi.pipeline`. Увеличенный GPU-кандидат не является обязательным условием использования уже готового submission.
